# 03 — Baselines comparison

The mandatory table compares Zero-shot Claude, DistilBERT-only and the Hybrid system. `check_availability` reports which ones can run here; the ablation of the confidence score runs on the recorded offline signals.

In [1]:
import asyncio, json, os
from pathlib import Path
os.environ.pop("ANTHROPIC_API_KEY", None)          # offline: deterministic fallback
os.environ["USE_LOCAL_CLASSIFIER"] = "false"
from src.config import Settings
from src.api.schemas import TicketIn
from src.eval.runner import build_offline_pipeline, run_pipeline_eval, load_cases, format_report
settings = Settings(anthropic_api_key=None, use_local_classifier=False)
pipeline = asyncio.run(build_offline_pipeline(settings))
print("classifier:", pipeline.classifier.name, "| llm_enabled:", pipeline.llm_enabled)

classifier: keyword_heuristic | llm_enabled: False


In [2]:
from src.eval.baselines import check_availability, ablate_confidence
avail = check_availability(settings)
print("llm:", avail.llm, "| hf_classifier:", avail.hf_classifier)
for k, v in avail.reasons.items():
    print(f"  {k}: {v}")

llm: False | hf_classifier: False
  llm: requiere ANTHROPIC_API_KEY
  hf_classifier: USE_LOCAL_CLASSIFIER=false


In [3]:
cases = load_cases()
report = asyncio.run(run_pipeline_eval(pipeline, cases))
print(f"{'variant':<22} agreement   auto  false_esc  false_auto")
for row in ablate_confidence(report, cases, settings):
    print(f"{row.variant:<22} {row.decision_agreement:>8.1%} {row.auto_resolve_rate:>6.1%} "
          f"{row.false_escalation_rate:>9.1%} {row.false_auto_resolve_rate:>10.1%}")

variant                agreement   auto  false_esc  false_auto
full                      49.1%   0.0%     17.9%       0.0%
no_similarity             54.6%   0.0%      4.2%       0.0%
no_sentiment_penalty      49.1%   0.0%     17.9%       0.0%
intent_only               54.6%   0.0%      4.2%       0.0%


With the hashed encoder the similarity term is noise: removing it raises decision agreement and lowers false escalations. The thresholds are calibrated for the semantic encoder (`all-mpnet-base-v2`), which is why the fallback is never reported as the system.